# 1.얼굴 검출

In [ ]:
# MediaPipe/OpenCV 설치 명령
# 설치된 환경은 생략 가능
%pip install --upgrade mediapipe opencv-contrib-python
# uv add --upgrade mediapipe opencv-contrib-python

In [2]:
# OpenCV: 웹캠 입력/색상 변환/화면 출력
import cv2
# MediaPipe: 이미지 래핑/모델 실행
import mediapipe as mp

In [3]:
# MediaPipe 버전 확인
print(mp.__version__)

# Tasks API 사용 가능 여부 확인
print(hasattr(mp, "tasks"))

0.10.35
True


In [4]:
# 모델 경로/Tasks API 클래스 준비
from pathlib import Path
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [5]:
# 모델 저장 폴더
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# 얼굴 박스 감지 모델
FACE_DETECTOR_MODEL = MODEL_DIR / "blaze_face_short_range.tflite"

# 얼굴 랜드마크 모델
FACE_LANDMARKER_MODEL = MODEL_DIR / "face_landmarker.task"

# 공식 모델 URL
FACE_DETECTOR_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
FACE_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"

In [6]:
def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return

    # 모델이 없을 때만 다운로드
    print(f"모델 다운로드 중: {model_path}")
    urllib.request.urlretrieve(model_url, model_path)


# 첫 실행 시 모델 자동 다운로드
download_model(FACE_DETECTOR_MODEL, FACE_DETECTOR_MODEL_URL)
download_model(FACE_LANDMARKER_MODEL, FACE_LANDMARKER_MODEL_URL)

모델 다운로드 중: models\blaze_face_short_range.tflite
모델 다운로드 중: models\face_landmarker.task


In [7]:
# Tasks API 클래스 별칭
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
FaceDetector = vision.FaceDetector
FaceDetectorOptions = vision.FaceDetectorOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
FaceLandmarksConnections = vision.FaceLandmarksConnections
mp_drawing = vision.drawing_utils

In [ ]:
def create_face_detector(min_detection_confidence=0.5):
    # 얼굴 박스/keypoint 감지기 생성
    # 셀을 따로 실행해도 필요한 Tasks API 클래스가 준비되도록 한 번 더 가져옴
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    options = vision.FaceDetectorOptions(
        # base_options는 어떤 모델 파일을 사용할지 지정
        base_options=python.BaseOptions(model_asset_path=str(FACE_DETECTOR_MODEL)),
        running_mode=vision.RunningMode.VIDEO,   # 웹캠처럼 시간 순서가 있는 연속 프레임 처리할 때 사용
        min_detection_confidence=min_detection_confidence  # 감지 신뢰도
    )
    return vision.FaceDetector.create_from_options(options)


def normalized_to_pixel_coordinates(normalized_x, normalized_y, image_width, image_height):
    # 정규화 좌표 -> 픽셀 좌표 변환
    # MP의 keypoint 좌표는 0~1 사이의 정규화 좌표. OpenCV에서 점을 그리려면 픽셀 좌표로 바꿔야 함.
    if not (0 <= normalized_x <= 1 and 0 <= normalized_y <= 1):
        return None

    # 이미지 범위를 벗어나지 않도록 마지막 픽셀 인덱스로 제한
    x = min(int(normalized_x * image_width), image_width - 1)
    y = min(int(normalized_y * image_height), image_height - 1)
    return x, y


def draw_face_detections(image, detection_result):
    # 감지 결과 표시
    height, width, _ = image.shape

    # 얼굴이 감지되지 않은 프레임은 그대로 반환
    if not detection_result.detections:
        return image

    # 얼굴을 감싸는 바운딩 박스
    for detection in detection_result.detections:
        bbox = detection.bounding_box
        start_point = (int(bbox.origin_x), int(bbox.origin_y))
        end_point = (int(bbox.origin_x + bbox.width), int(bbox.origin_y + bbox.height))
        cv2.rectangle(image, start_point, end_point, (0, 0, 255), 2)

        # 눈, 코, 입 등 얼굴의 대표 지점
        for keypoint in detection.keypoints:
            keypoint_px = normalized_to_pixel_coordinates(keypoint.x, keypoint.y, width, height)
            if keypoint_px:
                cv2.circle(image, keypoint_px, 3, (0, 255, 0), -1)

        # 얼굴 감지 신뢰도 표시
        if detection.categories:
            score = detection.categories[0].score
            cv2.putText(
                image,
                f"face {score:.2f}",
                (int(bbox.origin_x), max(20, int(bbox.origin_y) - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )

    return image


In [11]:
# 0번 카메라 열기
cap = cv2.VideoCapture(0)

# FPS 없으면 30 사용
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

# 모델 리소스 자동 정리용 with 문
with create_face_detector(min_detection_confidence=0.5) as face_detector:
    while cap.isOpened():
        # 웹캠 프레임 읽기
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe 입력 이미지 생성
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # 프레임 번호 -> timestamp 변환
        # VIDEO 모드에서는 프레임마다 증가하는 timestamp가 필요
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 현재 프레임 얼굴 감지
        # face_detector : 웹캠/동영상처럼 연속된 프레임을 처리할 때 쓰는 감지 함
        detection_result = face_detector.detect_for_video(mp_image, timestamp_ms)

        # 얼굴 박스/keypoint 표시
        # 감지된 얼굴 바운딩박스와 keypoing를 원본 이미지 위에 그림.
        draw_face_detections(image, detection_result)

        # 결과 화면 출력
        cv2.imshow('frame', image)

        # q 입력 시 종료
        if cv2.waitKey(1) == ord('q'):
            break

# 카메라/창 정리
cap.release()
cv2.destroyAllWindows()

# 2.얼굴 전체 메시 그려보기

In [12]:
# 랜드마크 점/선 스타일
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

In [ ]:
def create_face_landmarker(num_faces=1):
    # 얼굴 랜드마크 검출기 생성
    # 이 셀만 다시 실행해도 필요한 Tasks API 클래스가 준비되도록 함수 안에서 import
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(FACE_LANDMARKER_MODEL)),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=num_faces,                # 한 프레임에서 최대 몇 명의 얼굴을 찾을지
        min_face_detection_confidence=0.5,  # 얼굴 검출 최소 신뢰도
        min_face_presence_confidence=0.5,   # 얼굴이 부분적으로 가려졌을 때 결과 안정성에 영향을 줌
        min_tracking_confidence=0.5         # 값이 높으면 안정적인 추적만 유지
    )
    return vision.FaceLandmarker.create_from_options(options)


In [ ]:
# 얼굴 메시 실시간 표시 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # timestamp 계산
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 얼굴 랜드마크 검출
        result =  face_landmarker.detect_for_video(mp_image, timestamp_ms)

        # result.face_landmarks : 얼굴마다 랜드마크 리스트를 담고 있음
        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # 얼굴 전체 메시 연결선 그리기
                mp_drawing.draw_landmarks(
                    image=image, # 랜드마크를 그릴 이미지
                    landmark_list=face_landmarks, # 검출된 얼굴 랜드마크 좌표들
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                    landmark_drawing_spec=drawing_spec,     # 점 스타일
                    connection_drawing_spec= drawing_spec   # 선 스타일
                )

        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 3.눈 부위의 랜드마크 그려보기

In [14]:
# 눈 주변 랜드마크 인덱스
LEFT_EYE_INDICES = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
RIGHT_EYE_INDICES = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]

In [15]:
# 눈 주변 점 표시 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 얼굴 랜드마크 검출
        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # 정규화 좌표 -> 픽셀 좌표 변환 후 눈 점 표시
                for idx in LEFT_EYE_INDICES:
                    point = face_landmarks[idx]
                    x = int(point.x * image.shape[1])
                    y = int(point.y * image.shape[0])
                    cv2.circle(image, (x, y), 1, (255, 0, 0), -1)

                for idx in RIGHT_EYE_INDICES:
                    point = face_landmarks[idx]
                    x = int(point.x * image.shape[1])
                    y = int(point.y * image.shape[0])
                    cv2.circle(image, (x, y), 1, (0, 0, 255), -1)

        # 출력 직전 좌우 반전
        image = cv2.flip(image, 1)
        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 4.기본 제공 랜드마크로 그려보기

In [16]:
# 눈 연결선 스타일
# OpenCV 색상은 BGR 순서
drawing_spec1 = mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0, 255, 0))
drawing_spec2 = mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(255, 0, 0))

In [ ]:
# 기본 눈 연결 정보 표시 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # 왼쪽 눈 연결선만 그림
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_LEFT_EYE,
                    landmark_drawing_spec=None,             # 점 그리지 x
                    connection_drawing_spec=drawing_spec1   # 초록색
                )
                # 오른쪽 눈 연결선만 그림
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_EYE,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=drawing_spec2   # 파란색
                )       

        image = cv2.flip(image, 1)
        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

[실습-자유주제]
- 눈썹부위 랜드마크 그려보기
- 코 부위 랜드마크
- 입술 랜드마크
- 안경씌우기
- 졸음방지

In [25]:
# 졸음 방지: 눈을 일정 시간 이상 감으면 알림음 울리기
import math
import time
import winsound

# EAR 계산용 눈 랜드마크 인덱스
# [눈 바깥쪽, 위1, 위2, 눈 안쪽, 아래1, 아래2]
LEFT_EYE_EAR = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_EAR = [362, 385, 387, 263, 373, 380]

# 값이 작을수록 눈을 감은 상태
EAR_THRESHOLD = 0.20

# 이 시간 이상 눈을 감고 있으면 알림
CLOSED_SECONDS_THRESHOLD = 1.0

# 알림음 반복 간격
ALARM_INTERVAL = 1.0


def distance(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)


def eye_aspect_ratio(face_landmarks, indices):
    p1, p2, p3, p4, p5, p6 = [face_landmarks[i] for i in indices]
    vertical_1 = distance(p2, p6)
    vertical_2 = distance(p3, p5)
    horizontal = distance(p1, p4)

    if horizontal == 0:
        return 0

    return (vertical_1 + vertical_2) / (2.0 * horizontal)


def draw_eye_points(image, face_landmarks, indices, color):
    height, width, _ = image.shape
    for idx in indices:
        point = face_landmarks[idx]
        x = int(point.x * width)
        y = int(point.y * height)
        cv2.circle(image, (x, y), 2, color, -1)


cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

eye_closed_start = None
last_alarm_time = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        status_text = 'No face'
        status_color = (0, 255, 255)

        if result.face_landmarks:
            face_landmarks = result.face_landmarks[0]

            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE_EAR)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE_EAR)
            ear = (left_ear + right_ear) / 2.0

            draw_eye_points(image, face_landmarks, LEFT_EYE_EAR, (255, 0, 0))
            draw_eye_points(image, face_landmarks, RIGHT_EYE_EAR, (0, 0, 255))

            now = time.time()

            if ear < EAR_THRESHOLD:
                if eye_closed_start is None:
                    eye_closed_start = now

                closed_time = now - eye_closed_start

                if closed_time >= CLOSED_SECONDS_THRESHOLD:
                    status_text = 'DROWSINESS WARNING!'
                    status_color = (0, 0, 255)

                    if now - last_alarm_time >= ALARM_INTERVAL:
                        winsound.PlaySound('SystemExclamation', winsound.SND_ALIAS | winsound.SND_ASYNC)
                        last_alarm_time = now
                else:
                    status_text = 'Eyes closed'
                    status_color = (0, 165, 255)
            else:
                eye_closed_start = None
                status_text = 'Eyes open'
                status_color = (0, 255, 0)

            cv2.putText(
                image,
                f'EAR: {ear:.2f}',
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

        cv2.putText(
            image,
            status_text,
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            status_color,
            2,
            cv2.LINE_AA,
        )

        image = cv2.flip(image, 1)
        cv2.imshow('Drowsiness Detector', image)

        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
